# Telco Customer Churn – Capstone Analysis
**Business problem:** A telecom company is losing customers. We build models to predict which customers will churn so the retention team can target them with offers.

**Success metrics:** Accuracy is misleading because only ~27% of customers churn, so we focus on **F1-score, recall (catching churners) and ROC-AUC**.

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 columns).

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# --- Paths (works locally and in Colab if the repo root is the working directory) ---
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())
DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
FIG_DIR = "reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

from src import data_preprocessing as dp

## 1. Load and inspect the data

In [ ]:
raw = dp.load_data(DATA_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.info()
print("\nBlank TotalCharges:", (raw["TotalCharges"].astype(str).str.strip() == "").sum())
print("Duplicate rows:", raw.drop(columns="customerID").duplicated().sum())

## 2. Exploratory Data Analysis

In [ ]:
clean = dp.clean_data(raw)
churn_rate = (clean["Churn"] == "Yes").mean()
print(f"Overall churn rate: {churn_rate:.1%}")
ax = clean["Churn"].value_counts().plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_title("Class distribution (imbalanced)"); plt.xticks(rotation=0)
plt.savefig(f"{FIG_DIR}/01_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = clean.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
    rate.plot(kind="barh", ax=ax, color="#C44E52")
    ax.set_title(f"Churn rate by {col}"); ax.set_xlabel("Churn rate")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_churn_by_category.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=clean, x=col, hue="Churn", kde=True, ax=ax, element="step")
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_numeric_distributions.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Boxplots to visually check outliers
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.boxplot(data=clean, y=col, x="Churn", ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Features most correlated with churn
encoded = dp.encode_features(dp.engineer_features(clean))
corr = encoded.corr()["Churn"].drop("Churn").sort_values()
top = pd.concat([corr.head(8), corr.tail(8)])
plt.figure(figsize=(7, 6))
top.plot(kind="barh", color=["#4C72B0" if v < 0 else "#C44E52" for v in top])
plt.title("Features most correlated with churn")
plt.savefig(f"{FIG_DIR}/04_correlations.png", dpi=150, bbox_inches="tight"); plt.show()

### EDA takeaways
*(Edit this after running – replace with what YOUR charts show.)*
- Churn rate is about 27%, so the classes are imbalanced.
- Month-to-month contract customers churn far more than one/two-year customers.
- Customers with short tenure churn most; loyalty rises with tenure.
- Fiber optic and electronic-check customers show higher churn.
- Customers with more add-on services (security, tech support) churn less.

## 3. Preprocessing and feature engineering
All logic lives in `src/data_preprocessing.py`: cleaning, new features (`num_addons`, `avg_monthly_spend`, `tenure_group`), encoding, IQR outlier capping and scaling (fit on the training set only).

In [ ]:
X_train, X_test, y_train, y_test, scaler = dp.preprocess(DATA_PATH)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Churn rate train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))
X_train.head()

## 4. Modeling
We compare a **Logistic Regression baseline** against a **tuned Random Forest**, both using `class_weight='balanced'` to handle the imbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, f1_score, recall_score, precision_score, accuracy_score)

# Baseline
baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
baseline.fit(X_train, y_train)

In [ ]:
# Tuned Random Forest
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_dist = {
    "n_estimators": [200, 300, 500],
    "max_depth": [5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}
search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    param_distributions=param_dist, n_iter=20, scoring="f1", cv=cv, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
rf = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))

## 5. Evaluation

In [ ]:
models = {"Logistic Regression (baseline)": baseline, "Random Forest (tuned)": rf}
rows = []
for name, m in models.items():
    pred = m.predict(X_test); proba = m.predict_proba(X_test)[:, 1]
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, proba)})
results = pd.DataFrame(rows).set_index("Model").round(3)
results.to_csv("reports/model_comparison.csv")
results

In [ ]:
for name, m in models.items():
    print("=" * 60); print(name); print("=" * 60)
    print(classification_report(y_test, m.predict(X_test), target_names=["Stay", "Churn"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, m) in zip(axes, models.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(X_test)),
                           display_labels=["Stay", "Churn"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_confusion_matrices.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for name, m in models.items():
    p = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves"); plt.legend()
plt.savefig(f"{FIG_DIR}/06_roc_curves.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values().tail(12)
plt.figure(figsize=(7, 5)); imp.plot(kind="barh", color="#4C72B0")
plt.title("Top 12 features – Random Forest")
plt.savefig(f"{FIG_DIR}/07_feature_importance.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Findings, limitations and future work
*(Edit with your real numbers after running.)*

**Findings**
- Compare the tuned Random Forest with the baseline using recall, F1 and ROC-AUC in the table above.
- Top churn drivers: contract type, tenure, monthly charges, internet service type.

**Business recommendations**
1. Offer discounts or upgrades to move month-to-month customers onto 1–2 year contracts.
2. Run an onboarding/retention programme for customers in their first 12 months.
3. Bundle tech support and online security, since customers with add-ons churn less.
4. Review fiber-optic pricing/quality and electronic-check payment friction.

**Limitations**
- Single snapshot of data (no time dimension); class imbalance; no cost-of-retention data.

**Future work**
- Try XGBoost/LightGBM, probability calibration, threshold tuning based on retention cost, SHAP explanations.